## Differential Solar Rotation 
### Task 3

Cortés, I., (2024) for the AstroLab Summer term, Heidelberg University

iliana.cortes@h-its.org


## Introduction

In this notebook, we will explore some key concepts related to solar observatories, Lagrangian points, and the structure of the Sun. 

### Questions

1. **What are the Lagrangian points? Why was SOHO put onto L1?**

   - Lagrangian points (or libration points) are **equilibrium points** where the gravitational forces of two massive bodies (like the Earth and the Sun) and the centrifugal force balance. Small objects can remain stationary relative to the two larger bodies at these points. 

2. **Why was SOHO put in a halo orbit around L1 instead of directly at L1?**

   - SOHO was placed in a halo orbit around L1 to maintain continuous, stable observation of the Sun while minimizing the need for propulsion. Unlike a direct placement at L1, a halo orbit provides a **stable, quasi-periodic trajectory** that reduces the need for active station-keeping.

   - **Reference**: [Halo Orbits Around Lagrangian Points](https://www.fossilhunters.xyz/spaceflight/halo-orbits-around-lagrangian-points.html) (external link for further reading)

   - **Types of Orbits Around Lagrangian Points**:
     - **Lissajous Orbit**: A quasi-periodic orbit around a Lagrangian point, named after Jules Antoine Lissajous. These orbits combine components in the plane of the two primary bodies and components perpendicular to this plane.
     - **Lyapunov Orbit**: A curved path in the plane of the two primary bodies, entirely two-dimensional.
     - **Halo Orbit**: A periodic, three-dimensional orbit around a Lagrangian point.

3. **What distinguishes the three outer layers of the Sun—the photosphere, chromosphere, and corona—from each other?**

   - **Temperature**: 
     - The photosphere is the coolest (~5,500°C).
     - The chromosphere becomes progressively hotter.
     - The corona reaches extreme temperatures (~1-3 million °C).
   - **Visibility**:
     - The photosphere is the visible "surface" of the Sun.
     - The chromosphere and corona are visible only during eclipses or with special instruments (e.g., ultraviolet or X-rays).
   - **Density**:
     - The photosphere is the densest, followed by the chromosphere.
     - The corona is the least dense and most diffuse.
   - **Activity**:
     - The chromosphere shows dynamic phenomena like **spicules** and **prominences**.
     - The corona hosts high-energy events such as **solar flares** and the **solar wind**.




   
<div style="text-align: center;">
    <img src="https://pwonlyias.com/wp-content/uploads/2023/11/picture9-655865a99d1a6.webp" alt="Solar Layers" width="400"/>
</div>

4. **Why is the temperature gradient in the Solar Corona positive (i.e., the farther out, the higher the temperature)?**

   The positive temperature gradient in the corona is a well-known puzzle in solar physics, termed the **coronal heating problem**. Despite being farther from the Sun’s core, the corona reaches temperatures of millions of degrees. This is counterintuitive, as one might expect temperatures to decrease with distance.

   - **Magnetic Fields**: The Sun's magnetic fields transport energy outward through waves and reconnection events.
   - **Wave Heating**: Magnetic and acoustic waves transfer energy upwards and release it in the corona.
   - **Nanoflares**: Small-scale bursts of energy from magnetic reconnection may contribute to localized heating.
   - **Low Density**: The corona’s extremely low density makes traditional heat conduction inefficient, allowing heat to accumulate.
   - **Energy Dissipation**: The interaction of waves and magnetic fields in the corona results in the conversion of magnetic energy into heat.

By understanding these concepts, we gain insight into why solar observatories like SOHO are strategically positioned and how the solar atmosphere functions at various layers.


## Determination of the differential solar rotation using SOHO data


In [2]:
# Libraries

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as ct
from astropy import units as u


### Data acquisition

The first step is to extract the position $(X,Y)$ of the sunspot, taking into account the center of the Sun in image coordinates, as well as the radius of the Sun. The procedure is described in the task. 

<div style="text-align: center;">
    <img src="https://i.ibb.co/ZRNZDckY/09-01-05.gif" alt="Solar Layers" width="400"/>
</div>

In [13]:
data_09_01_05 = {
    'OBS_TIME': ['2005-01-09T06:23:32.198Z', '2005-01-09T11:11:32.200Z', '2005-01-09T15:59:32.202Z', '2005-01-09T23:59:32.206Z'], # observation time
    'POS_X': [75, 87, 99, 121], # x coordinate of the sunspot 
    'POS_Y': [469, 471, 472, 476], # y coordinate of the sunspot
    'SUN_X': [5.127242431641E+02, 5.127218017578E+02, 5.125944824219E+02, 5.126706237793E+02], # x coordinate of the solar centre
    'SUN_Y': [5.119660949707E+02, 5.119720153809E+02, 5.119750671387E+02, 5.119614868164E+02], # y coordinate of the solar centre
    'RAD_SUN': [4.956781079938E+02, 4.956769024329E+02, 4.956756248224E+02, 4.956733357482E+02], # radius of the sun
    # Solar ephemeris
    # 'ALPHA': [1952.36, 1952.35, 1952.33, 1952.3], # angular diameter of the Sun arcsec
    'P_ANGLE': [-1.999, -2.095, -2.190, -2.350], #position angle of the rotation axis
    'B_0': [-3.969, -3.991, -4.012, -4.048], # latitude of the solar disk centre
    'L_0': [265.014, 262.380, 259.747, 255.357] # longitude of the solar disk centre
}


data_09_01_05 = pd.DataFrame(data_09_01_05)

data_09_01_05['OBS_TIME'] = pd.to_datetime(data_09_01_05['OBS_TIME'])


print(data_09_01_05)

                          OBS_TIME  POS_X  POS_Y       SUN_X       SUN_Y  \
0 2005-01-09 06:23:32.198000+00:00     75    469  512.724243  511.966095   
1 2005-01-09 11:11:32.200000+00:00     87    471  512.721802  511.972015   
2 2005-01-09 15:59:32.202000+00:00     99    472  512.594482  511.975067   
3 2005-01-09 23:59:32.206000+00:00    121    476  512.670624  511.961487   

      RAD_SUN  P_ANGLE    B_0      L_0  
0  495.678108   -1.999 -3.969  265.014  
1  495.676902   -2.095 -3.991  262.380  
2  495.675625   -2.190 -4.012  259.747  
3  495.673336   -2.350 -4.048  255.357  


In [20]:
data_17_01_05 = {
    'OBS_TIME': ['2005-01-17T06:23:32.287Z', '2005-01-17T11:11:32.289Z', '2005-01-17T15:59:32.292Z'], # observation time
    'POS_X': [666, 688, 712], # x coordinate of the sunspot 
    'POS_Y': [646, 646, 637], # y coordinate of the sunspot
    'SUN_X': [5.128960571289E+02, 5.128827819824E+02,5.128643493652E+02], # x coordinate of the solar centre
    'SUN_Y': [5.119730834961E+02, 5.119662170410E+02, 5.119624023438E+02], # y coordinate of the solar centre
    'RAD_SUN': [4.955751029422E+02, 4.955711294998E+02, 4.955670897308E+02], # radius of the sun
    # Solar ephemeris
    # 'ALPHA': [1951.6, 1951.58, 1951.55], # angular diameter of the Sun arcsec
    'P_ANGLE': [-5.775, -5.868, -5.960], #position angle of the rotation axis
    'B_0': [-4.789, -4.808, -4.827], # latitude of the solar disk centre
    'L_0': [159.673, 157.040, 154.406] # longitude of the solar disk centre
}


data_17_01_05 = pd.DataFrame(data_17_01_05)

data_17_01_05['OBS_TIME'] = pd.to_datetime(data_17_01_05['OBS_TIME'])


print(data_17_01_05)

                          OBS_TIME  POS_X  POS_Y       SUN_X       SUN_Y  \
0 2005-01-17 06:23:32.287000+00:00    666    646  512.896057  511.973083   
1 2005-01-17 11:11:32.289000+00:00    688    646  512.882782  511.966217   
2 2005-01-17 15:59:32.292000+00:00    712    637  512.864349  511.962402   

      RAD_SUN  P_ANGLE    B_0      L_0  
0  495.575103   -5.775 -4.789  159.673  
1  495.571129   -5.868 -4.808  157.040  
2  495.567090   -5.960 -4.827  154.406  


In [23]:
data_27_12_09 = {
    'OBS_TIME': ['2009-12-27T06:23:31.882Z', '2009-12-27T11:32:31.887Z', '2009-12-27T15:59:31.892Z'], # observation time
    'POS_X': [219, 236, 250], # x coordinate of the sunspot 
    'POS_Y': [293, 293, 294], # y coordinate of the sunspot
    'SUN_X': [5.127313232422E+02, 5.127303771973E+02, 5.127270202637E+02], # x coordinate of the solar centre
    'SUN_Y': [5.116305541992E+02, 5.116247329712E+02, 5.116341857910E+02], # y coordinate of the solar centre
    'RAD_SUN': [4.963766681838E+02, 4.963869073162E+02, 4.963956568728E+02], # radius of the sun
    # Solar ephemeris
    # 'ALPHA': [1952.33, 1952.35, 1952.37], # angular diameter of the Sun arcsec
    'P_ANGLE': [4.389, 4.288, 4.199], #position angle of the rotation axis
    'B_0': [-2.442, -2.468, -2.490], # latitude of the solar disk centre
    'L_0': [95.160, 92.407, 89.965] # longitude of the solar disk centre
}


data_27_12_09 = pd.DataFrame(data_27_12_09)

data_27_12_09['OBS_TIME'] = pd.to_datetime(data_27_12_09['OBS_TIME'])


print(data_27_12_09)

                          OBS_TIME  POS_X  POS_Y       SUN_X       SUN_Y  \
0 2009-12-27 06:23:31.882000+00:00    219    293  512.731323  511.630554   
1 2009-12-27 11:32:31.887000+00:00    236    293  512.730377  511.624733   
2 2009-12-27 15:59:31.892000+00:00    250    294  512.727020  511.634186   

      RAD_SUN  P_ANGLE    B_0     L_0  
0  496.376668    4.389 -2.442  95.160  
1  496.386907    4.288 -2.468  92.407  
2  496.395657    4.199 -2.490  89.965  


In [24]:
data_02_01_2010 = {
    'OBS_TIME': ['2010-01-02T07:58:32.042Z', '2010-01-02T12:07:32.047Z', '2010-01-02T18:41:32.055Z', '2010-01-02T23:59:32.061Z'], # observation time
    'POS_X': [810, 822, 840, 855], # x coordinate of the sunspot 
    'POS_Y': [298, 298, 297, 296], # y coordinate of the sunspot
    'SUN_X': [5.127382532756E+02, 5.127405293783E+02, 5.127350158691E+02, 5.127363891602E+02], # x coordinate of the solar centre
    'SUN_Y': [5.116251195272E+02, 5.116242980957E+02, 5.116033630371E+02, 5.116225280762E+02], # y coordinate of the solar centre
    'RAD_SUN': [4.966171648737E+02, 4.966224656774E+02, 4.966306693200E+02, 4.966371254274E+02], # radius of the sun
    # Solar ephemeris
    # 'ALPHA': [1952.63, 1952.63, 1952.63, 1952.63], # angular diameter of the Sun arcsec
    'P_ANGLE': [1.456, 1.372, 1.239, 1.132], #position angle of the rotation axis
    'B_0': [-3.161, -3.181, -3.213, -3.238 ], # latitude of the solar disk centre
    'L_0': [15.337, 13.059, 9.456, 6.547] # longitude of the solar disk centre
}


data_02_01_2010 = pd.DataFrame(data_02_01_2010)

data_02_01_2010['OBS_TIME'] = pd.to_datetime(data_02_01_2010['OBS_TIME'])


print(data_02_01_2010)

                          OBS_TIME  POS_X  POS_Y       SUN_X       SUN_Y  \
0 2010-01-02 07:58:32.042000+00:00    810    298  512.738253  511.625120   
1 2010-01-02 12:07:32.047000+00:00    822    298  512.740529  511.624298   
2 2010-01-02 18:41:32.055000+00:00    840    297  512.735016  511.603363   
3 2010-01-02 23:59:32.061000+00:00    855    296  512.736389  511.622528   

      RAD_SUN  P_ANGLE    B_0     L_0  
0  496.617165    1.456 -3.161  15.337  
1  496.622466    1.372 -3.181  13.059  
2  496.630669    1.239 -3.213   9.456  
3  496.637125    1.132 -3.238   6.547  


---

### Heliographic Coordinates 

#### The naive two-dimensional approach

Considering, a low latitude, we can make the following approximation:

<div style="text-align: center;">
    <!--img src="https://i.ibb.co/vvCsBjyy/sunspot.png" width="600"/-->
    <img src="https://i.ibb.co/v6RzDwJQ/sunspot-apporx.png"   width="600"/>
    
</div>


where 

- $\lambda$ is the latitude 
- $l$ is the longitude
- $(x,y)$ the coordinates of the sunspot
- $R_{Sun}$ the radius of the Sun
- $(x_0, y_0)$ are the coordinates of the center


Then we can calculate 

$$\lambda = \arcsin\left(\frac{y-y_0}{R_{Sun}} \right)$$

$$l = \arcsin\left(\frac{x-x_0}{R_{Sun}} \right)$$



---

In [47]:
#### Approximate coordinates 

def helio_coordinates_approx(row):
   # Small angle approximation for the latitude
    # small_angle = False
    B0 = np.radians(row['B_0'])
    L0 = np.radians(row['L_0'])

    x = (row['POS_X'] - row['SUN_X']) # pixel coordinate x 
    y = (row['POS_Y'] - row['SUN_Y']) # pixel coordinate y

    rho = row['RAD_SUN'] #np.sqrt(x**2 + y**2) # radius

    # latitude
    latitude = np.arcsin(y/rho) #+ B0

    # longitude
    longitude = np.arcsin(x/rho) #+ L0

   
    return pd.Series({'LATITUDE': np.degrees(latitude), 'LONGITUDE': np.degrees(longitude)})


In [48]:
data_09_01_05[['LATITUDE', 'LONGITUDE']] = data_09_01_05.apply(helio_coordinates_approx, axis=1)

# Print table
print(data_09_01_05[['OBS_TIME', 'POS_X', 'POS_Y', 'LATITUDE', 'LONGITUDE']])

                          OBS_TIME  POS_X  POS_Y  LATITUDE  LONGITUDE
0 2005-01-09 06:23:32.198000+00:00     75    469 -4.972722 -62.016366
1 2005-01-09 11:11:32.200000+00:00     87    471 -4.741405 -59.189893
2 2005-01-09 15:59:32.202000+00:00     99    472 -4.625793 -56.553989
3 2005-01-09 23:59:32.206000+00:00    121    476 -4.160509 -52.202235


In [49]:
data_17_01_05[['LATITUDE', 'LONGITUDE']] = data_17_01_05.apply(helio_coordinates_approx, axis=1)

# Print table
print(data_17_01_05[['OBS_TIME', 'POS_X', 'POS_Y', 'LATITUDE', 'LONGITUDE']])


                          OBS_TIME  POS_X  POS_Y   LATITUDE  LONGITUDE
0 2005-01-17 06:23:32.287000+00:00    666    646  15.690882  17.995480
1 2005-01-17 11:11:32.289000+00:00    688    646  15.691835  20.693238
2 2005-01-17 15:59:32.292000+00:00    712    637  14.614376  23.692874


In [50]:
data_27_12_09[['LATITUDE', 'LONGITUDE']] =data_27_12_09.apply(helio_coordinates_approx, axis=1)

print(data_27_12_09[['OBS_TIME', 'POS_X', 'POS_Y', 'LATITUDE', 'LONGITUDE']])

                          OBS_TIME  POS_X  POS_Y   LATITUDE  LONGITUDE
0 2009-12-27 06:23:31.882000+00:00    219    293 -26.132783 -36.281354
1 2009-12-27 11:32:31.887000+00:00    236    293 -26.131455 -33.882342
2 2009-12-27 15:59:31.892000+00:00    250    294 -26.003680 -31.956103


In [51]:
data_02_01_2010[['LATITUDE', 'LONGITUDE']] =data_02_01_2010.apply(helio_coordinates_approx, axis=1)

print(data_02_01_2010[['OBS_TIME', 'POS_X', 'POS_Y', 'LATITUDE', 'LONGITUDE']])

                          OBS_TIME  POS_X  POS_Y   LATITUDE  LONGITUDE
0 2010-01-02 07:58:32.042000+00:00    810    298 -25.477750  36.767782
1 2010-01-02 12:07:32.047000+00:00    822    298 -25.477354  38.515439
2 2010-01-02 18:41:32.055000+00:00    840    297 -25.602089  41.221408
3 2010-01-02 23:59:32.061000+00:00    855    296 -25.732182  43.563839


---


### Calculating Angular Velocity, Synodic Period, and Sidereal Period

#### Angular Velocity

Angular velocity ($\omega$) is a measure of the rate of rotation and is typically expressed in radians per second (rad/s). It can be calculated using the formula:

$$
\omega = \frac{2\pi}{T}
$$

where $T$ is the period of rotation in seconds.

The angular velocity:

$$
\omega = \frac{\Delta \theta}{\Delta t}
$$

where $\Delta \theta$ is the change in angle in longitude (approximation) and $\Delta t$ is the change in time.

#### Sidereal Period

The sidereal period ($T_{sid}$) is the time taken for a celestial object to complete one full orbit around a central body (e.g., the Sun) with respect to distant stars. It can be calculated simply as:

$$
T_{sid} = \text{time period for one complete revolution}
$$

In the case of planets, this period can be related to their synodic period using the following formula:

$$
\frac{1}{T_{sid}} = \frac{1}{T_s} + \frac{1}{T_{Earth}}
$$

where:
- $T_{sid}$ is the sidereal period,
- $T_s$ is the synodic period,
- $T_{Earth}$ is the orbital period of the Earth (approximately 365.25 days).




In [52]:
# Synodic and Sidereal period

def velocity_and_period(df):
    results = []
    for i in range(len(df)):
        if i + 1 < len(df):
            time_1 = df.iloc[i]['OBS_TIME']
            time_2 = df.iloc[i + 1]['OBS_TIME']
            lon_1 = df.iloc[i]['LONGITUDE']
            lon_2 = df.iloc[i + 1]['LONGITUDE']
            lat_1 = df.iloc[i]['LATITUDE']  # Assumed latitude of the first spot
            
            time_diff_days = (time_2 - time_1).total_seconds() / (24 * 3600)
            
            angular_distance = abs(lon_2 - lon_1)
            
            
            
            angular_velocity = angular_distance / time_diff_days
            
            
            # Synodic period (days)
            synodic_period = 360 / angular_velocity
            
            # Sidereal period using the synodic formula
            sidereal_period_synodic = 1 / (1/synodic_period + 1/360)
            
            # Sidereal period using the theoretical solar model
            lat_radians = np.radians(lat_1)
            sidereal_period_theoretical = 25.38 + 2.77 * np.sin(lat_radians)**2
            
    
            
            results.append({
                'Assumed Latitude': lat_1,
                'Angular Distance (degrees)': angular_distance,
                'Angular Velocity (deg/day)': angular_velocity,
                'Synodic Period (days)': synodic_period,
                'Sidereal Period (Synodic formula)': sidereal_period_synodic,
                'Sidereal Period (Theoretical model)': sidereal_period_theoretical
            })
    
    return pd.DataFrame(results)


In [53]:

ang_vel_09_01 = velocity_and_period(data_09_01_05)

print(ang_vel_09_01)

   Assumed Latitude  Angular Distance (degrees)  Angular Velocity (deg/day)  \
0         -4.972722                    2.826473                   14.132362   
1         -4.741405                    2.635904                   13.179520   
2         -4.625793                    4.351753                   13.055257   

   Synodic Period (days)  Sidereal Period (Synodic formula)  \
0              25.473449                          23.790073   
1              27.315108                          25.388730   
2              27.575098                          25.613192   

   Sidereal Period (Theoretical model)  
0                            25.400813  
1                            25.398926  
2                            25.398016  


In [54]:

ang_vel_17_01 = velocity_and_period(data_17_01_05)

print(ang_vel_17_01)

   Assumed Latitude  Angular Distance (degrees)  Angular Velocity (deg/day)  \
0         15.690882                    2.697759                   13.488792   
1         15.691835                    2.999636                   14.998175   

   Synodic Period (days)  Sidereal Period (Synodic formula)  \
0              26.688824                          24.846791   
1              24.002920                          22.502567   

   Sidereal Period (Theoretical model)  
0                            25.582603  
1                            25.582627  


In [55]:

ang_vel_27_12 = velocity_and_period(data_27_12_09)

print(ang_vel_27_12)

   Assumed Latitude  Angular Distance (degrees)  Angular Velocity (deg/day)  \
0        -26.132783                    2.399012                   11.179859   
1        -26.131455                    1.926239                   10.388700   

   Synodic Period (days)  Sidereal Period (Synodic formula)  \
0              32.200764                          29.556993   
1              34.653036                          31.610280   

   Sidereal Period (Theoretical model)  
0                            25.917377  
1                            25.917326  


In [56]:

ang_vel_02_01 = velocity_and_period(data_02_01_2010)

print(ang_vel_02_01)

   Assumed Latitude  Angular Distance (degrees)  Angular Velocity (deg/day)  \
0        -25.477750                    1.747657                   10.106929   
1        -25.477354                    2.705969                    9.889833   
2        -25.602089                    2.342430                   10.607229   

   Synodic Period (days)  Sidereal Period (Synodic formula)  \
0              35.619130                          32.412201   
1              36.401017                          33.058357   
2              33.939119                          31.015155   

   Sidereal Period (Theoretical model)  
0                            25.892556  
1                            25.892541  
2                            25.897232  


---


#### Spherical Geomtery

Adapted from [Cortie (1897)](https://academic.oup.com/mnras/article/57/3/141/1007661). If we consider both high latitudes, and the tilt of the Sun. 

Where:
- $\lambda$ is the latitude 
- $l$ is the longitude
- $(x,y)$ the coordinates of the sunspot
- $R_{Sun}$ the radius of the Sun
- $(x_0, y_0)$ are the coordinates of the center
- $\phi$ is the position angle (tilt)

<div style="text-align: center;">
    <!--img src="https://i.ibb.co/vvCsBjyy/sunspot.png" width="600"/-->
    <img src="https://i.ibb.co/8Dz5XZjb/sunspot.png"  width="600"/>
    
</div>

We have

$$\bar{x} = x - x_0$$

$$\bar{y} = y - y_0$$

$$ST^2 = R_{Sun}^2 - (\bar{x}^2 + \bar{y}^2)$$

Then, the latitude: 

$$\sin \lambda = (\bar{y}\cos\phi + ST \sin \phi)/R_{Sun}$$


And the longitude:

$$\sin l = \frac{\bar{x}}{r}\cos \lambda$$

In [57]:
def helio_coordinates_new(row):

    R = row['RAD_SUN']

    P = np.radians(row['P_ANGLE'])
    B_0 = np.radians(row['B_0'])
    L_0 = np.radians(row['L_0'])

    # signed coordinates
    X = (row['POS_X'] - row['SUN_X']) #/ R
    Y = (row['POS_Y'] - row['SUN_Y']) #/ R  # flip image Y
    
    ST = np.sqrt(R**2 - (X**2 + Y*22))
    
    sin_B = (Y*np.cos(P) + ST*np.sin(P)) / R
    B = np.arcsin(sin_B)
    # B = B + B_0
    
    sin_L = (X/R) * np.cos(B)
    L = np.arcsin(sin_L)
    # L = L + L_0
    
    
    return pd.Series({'LATITUDE': np.degrees(B), 'LONGITUDE': np.degrees(L)})
    
    

In [58]:
data_09_01_05[['LATITUDE', 'LONGITUDE']] = data_09_01_05.apply(helio_coordinates_new, axis=1)

# Print table
print(data_09_01_05[['OBS_TIME', 'POS_X', 'POS_Y', 'LATITUDE', 'LONGITUDE']])

                          OBS_TIME  POS_X  POS_Y  LATITUDE  LONGITUDE
0 2005-01-09 06:23:32.198000+00:00     75    469 -5.919923 -61.446621
1 2005-01-09 11:11:32.200000+00:00     87    471 -5.823133 -58.697663
2 2005-01-09 15:59:32.202000+00:00     99    472 -5.841324 -56.106231
3 2005-01-09 23:59:32.206000+00:00    121    476 -5.608323 -51.850023


In [59]:
data_17_01_05[['LATITUDE', 'LONGITUDE']] = data_17_01_05.apply(helio_coordinates_new, axis=1)

# Print table
print(data_17_01_05[['OBS_TIME', 'POS_X', 'POS_Y', 'LATITUDE', 'LONGITUDE']])


                          OBS_TIME  POS_X  POS_Y   LATITUDE  LONGITUDE
0 2005-01-17 06:23:32.287000+00:00    666    646  10.021169  17.711760
1 2005-01-17 11:11:32.289000+00:00    688    646  10.024146  20.363211
2 2005-01-17 15:59:32.292000+00:00    712    637   9.003992  23.383418


In [60]:
data_27_12_09[['LATITUDE', 'LONGITUDE']] =data_27_12_09.apply(helio_coordinates_new, axis=1)

print(data_27_12_09[['OBS_TIME', 'POS_X', 'POS_Y', 'LATITUDE', 'LONGITUDE']])

                          OBS_TIME  POS_X  POS_Y   LATITUDE  LONGITUDE
0 2009-12-27 06:23:31.882000+00:00    219    293 -22.120208 -33.243309
1 2009-12-27 11:32:31.887000+00:00    236    293 -22.101814 -31.099318
2 2009-12-27 15:59:31.892000+00:00    250    294 -21.980728 -29.392959


In [61]:
data_02_01_2010[['LATITUDE', 'LONGITUDE']] =data_02_01_2010.apply(helio_coordinates_new, axis=1)

print(data_02_01_2010[['OBS_TIME', 'POS_X', 'POS_Y', 'LATITUDE', 'LONGITUDE']])

                          OBS_TIME  POS_X  POS_Y   LATITUDE  LONGITUDE
0 2010-01-02 07:58:32.042000+00:00    810    298 -24.165085  33.101242
1 2010-01-02 12:07:32.047000+00:00    822    298 -24.268122  34.589755
2 2010-01-02 18:41:32.055000+00:00    840    297 -24.549588  36.827005
3 2010-01-02 23:59:32.061000+00:00    855    296 -24.803363  38.725439


New periods:

In [62]:

ang_vel_09_01 = velocity_and_period(data_09_01_05)

print(ang_vel_09_01)

   Assumed Latitude  Angular Distance (degrees)  Angular Velocity (deg/day)  \
0         -5.919923                    2.748958                   13.744787   
1         -5.823133                    2.591433                   12.957161   
2         -5.841324                    4.256207                   12.768621   

   Synodic Period (days)  Sidereal Period (Synodic formula)  \
0              26.191749                          24.415409   
1              27.783864                          25.793211   
2              28.194118                          26.146410   

   Sidereal Period (Theoretical model)  
0                            25.409466  
1                            25.408514  
2                            25.408691  


In [63]:

ang_vel_17_01 = velocity_and_period(data_17_01_05)

print(ang_vel_17_01)

   Assumed Latitude  Angular Distance (degrees)  Angular Velocity (deg/day)  \
0         10.021169                    2.651451                   13.257252   
1         10.024146                    3.020207                   15.101032   

   Synodic Period (days)  Sidereal Period (Synodic formula)  \
0              27.154949                          25.250308   
1              23.839430                          22.358815   

   Sidereal Period (Theoretical model)  
0                            25.463876  
1                            25.463925  


In [64]:

ang_vel_27_12 = velocity_and_period(data_27_12_09)

print(ang_vel_27_12)

   Assumed Latitude  Angular Distance (degrees)  Angular Velocity (deg/day)  \
0        -22.120208                    2.143991                    9.991413   
1        -22.101814                    1.706359                    9.202832   

   Synodic Period (days)  Sidereal Period (Synodic formula)  \
0              36.030939                           32.75284   
1              39.118392                           35.28432   

   Sidereal Period (Theoretical model)  
0                             25.77276  
1                             25.77214  


In [65]:

ang_vel_02_01 = velocity_and_period(data_02_01_2010)

print(ang_vel_02_01)

   Assumed Latitude  Angular Distance (degrees)  Angular Velocity (deg/day)  \
0        -24.165085                    1.488513                    8.608263   
1        -24.268122                    2.237251                    8.176751   
2        -24.549588                    1.898433                    8.596676   

   Synodic Period (days)  Sidereal Period (Synodic formula)  \
0              41.820282                          37.467749   
1              44.027266                          39.229570   
2              41.876650                          37.512988   

   Sidereal Period (Theoretical model)  
0                            25.844201  
1                            25.847928  
2                            25.858169  
